# 28. Structured Outputs

**Tier:** Production & Safety
**Estimated time:** 35 minutes
**Prerequisites:** 12, 19
**Priority:** 🔴 Crucial — schema-constrained generation and validate→repair loops are the daily bread-and-butter of every LLM feature; unreliable JSON parsing is the #1 rookie production failure. *If skipped, revisit when:* n/a — this is 20 minutes of content that saves weeks of flaky downstream parsing.
**Source material:** Stanford Lecture 3 (prompting, function calling) — https://x.com/ajitcodes/status/2057043965317165490

## What You'll Learn
- Why "just ask for JSON" fails silently at scale, and what schema-constrained generation actually does about it
- Using the Anthropic tool-use mechanism to force a structured shape (not just hope for one)
- Validating model output against a Pydantic model
- The validate → repair loop pattern for the cases that still slip through

## Why This Matters
Every feature that turns an LLM's output into something a program consumes — a database row, an API response, a router decision — depends on the output actually parsing. Free-text "please respond in JSON" prompts break in exactly the moments that matter most: edge cases, long outputs, and anything the model finds mildly ambiguous. This notebook builds the pattern that makes structured output a guarantee, not a hope.


In [ ]:
import os, json
from pydantic import BaseModel, ValidationError, Field

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — live cells will be skipped.")


## Why "just ask for JSON" fails

The naive approach — put `"Respond only in JSON"` in the prompt and hope — works most of the time, which is exactly the problem: it fails unpredictably, and "unpredictably" is much worse for a production system than "always." Common failure modes: the model wraps the JSON in a markdown code fence, adds a conversational preamble ("Sure, here's the JSON:"), uses a slightly different key name than you expected, or omits a field it considered redundant.

In [ ]:
FREEFORM_SYSTEM = "Extract the person's name, age, and city from the text. Respond in JSON."

def ask_freeform(text):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    msg = client.messages.create(model=TEACH_MODEL, max_tokens=150, system=FREEFORM_SYSTEM,
                                  messages=[{"role": "user", "content": text}])
    return msg.content[0].text

sample = "Maria is 34 years old and lives in Lisbon."
raw = ask_freeform(sample)
print(repr(raw))

try:
    parsed = json.loads(raw)
    print("Parsed cleanly:", parsed)
except json.JSONDecodeError as e:
    print(f"Parse failed (the exact fragility this notebook fixes): {e}")


## Schema-constrained generation via tool use

Anthropic's tool-use mechanism isn't just for calling external functions — giving the model a tool schema with no real implementation behind it is a reliable way to force a specific output shape. The model MUST populate the schema's required fields to make the "call," which is a much stronger guarantee than a prompt instruction.

In [ ]:
EXTRACT_SCHEMA = {
    "name": "extract_person",
    "description": "Extract structured person data from text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "age": {"type": "integer"},
            "city": {"type": "string"},
        },
        "required": ["name", "age", "city"],
    },
}

def ask_structured(text):
    if not HAS_ANTHROPIC:
        return None
    msg = client.messages.create(
        model=TEACH_MODEL, max_tokens=150,
        system="Extract the requested fields from the text using the extract_person tool.",
        tools=[EXTRACT_SCHEMA],
        tool_choice={"type": "tool", "name": "extract_person"},   # force this exact tool
        messages=[{"role": "user", "content": text}],
    )
    for block in msg.content:
        if block.type == "tool_use":
            return block.input
    return None

structured = ask_structured(sample)
print(structured)


## Validating against a Pydantic model

The tool schema constrains the *shape*; Pydantic validates the *values* (types, ranges, custom rules) and gives you a typed object to work with instead of a raw dict. This is the boundary between "the model produced something schema-shaped" and "the model produced something your code can safely trust."

In [ ]:
class Person(BaseModel):
    name: str
    age: int = Field(ge=0, le=130)
    city: str

def validate_person(raw: dict) -> tuple[Person | None, str | None]:
    try:
        return Person(**raw), None
    except ValidationError as e:
        return None, str(e)

if structured:
    person, error = validate_person(structured)
    print(person if person else f"Validation failed: {error}")

# A deliberately bad extraction to show validation catching what the schema alone would miss.
bad_data = {"name": "Maria", "age": 999, "city": "Lisbon"}   # age out of range
person, error = validate_person(bad_data)
print(f"\nBad data caught: {error.splitlines()[0] if error else 'none'}" if error else person)


## The validate → repair loop

Even with a forced tool schema, edge cases slip through: an out-of-range value, a field the model filled with a placeholder, an argument that violates a business rule the schema can't express. The repair loop catches these by feeding the validation error back to the model and asking it to fix its own output — usually resolving in one extra round-trip.

In [ ]:
def extract_with_repair(text, max_attempts=2):
    messages = [{"role": "user", "content": text}]
    for attempt in range(max_attempts):
        if not HAS_ANTHROPIC:
            return None, "[skipped: no ANTHROPIC_API_KEY]"
        msg = client.messages.create(
            model=TEACH_MODEL, max_tokens=150,
            system="Extract the requested fields using the extract_person tool. Age must be 0-130.",
            tools=[EXTRACT_SCHEMA], tool_choice={"type": "tool", "name": "extract_person"},
            messages=messages,
        )
        tool_block = next((b for b in msg.content if b.type == "tool_use"), None)
        if tool_block is None:
            return None, "no tool call produced"
        person, error = validate_person(tool_block.input)
        if person:
            return person, None
        # Feed the validation error back for a repair attempt.
        messages.append({"role": "assistant", "content": msg.content})
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": tool_block.id,
             "content": f"Invalid: {error.splitlines()[0]}. Please call the tool again with a valid age.",
             "is_error": True},
        ]})
    return None, f"failed after {max_attempts} attempts"

# A prompt designed to tempt a bad age extraction (an ambiguous, oddly-phrased age).
tricky_text = "Maria has been around for what feels like a thousand years, and she lives in Lisbon."
person, error = extract_with_repair(tricky_text)
print(person if person else f"Repair loop result: {error}")


## Exercises

**Exercise 1 (Warm-up):** Add an `email` field (optional, validated as a proper email format) to `Person` / `EXTRACT_SCHEMA`, and confirm both the happy path and a missing-email case behave as expected.

**Exercise 2 (Apply):** Implement `extract_batch(texts: list[str]) -> list[Person]` that runs `extract_with_repair` over a list of texts and returns only the ones that validated successfully, printing how many were dropped.

**Exercise 3 (Extend):** Notebook 33 (CI for AI) checks prompts/schemas in CI. Sketch a unit test (using `pytest`-style assertions in comments) that would catch a regression where someone accidentally removes `"required": ["name", "age", "city"]` from `EXTRACT_SCHEMA`.


In [ ]:
# Exercise 1: Warm-up
# Task: Add an optional `email` field to Person and EXTRACT_SCHEMA, validated as an email format.
# Hint: Pydantic has an EmailStr type (pip install pydantic[email]) or use a simple regex Field.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement extract_batch(texts) -> list[Person], dropping and counting failures.
# Hint: reuse extract_with_repair per text; collect (person, error) pairs first, then filter.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch a CI unit test that fails if EXTRACT_SCHEMA's required fields list is weakened.
# Hint: assert on the schema dict directly — no API call needed for this kind of regression test.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
from typing import Optional
class PersonWithEmail(BaseModel):
    name: str
    age: int = Field(ge=0, le=130)
    city: str
    email: Optional[str] = None

EXTRACT_SCHEMA_V2 = {**EXTRACT_SCHEMA, "input_schema": {
    **EXTRACT_SCHEMA["input_schema"],
    "properties": {**EXTRACT_SCHEMA["input_schema"]["properties"], "email": {"type": "string"}},
}}

# Exercise 2
def extract_batch(texts):
    results = []
    n_failed = 0
    for t in texts:
        person, error = extract_with_repair(t)
        if person:
            results.append(person)
        else:
            n_failed += 1
    print(f"{len(results)} succeeded, {n_failed} dropped")
    return results

# Exercise 3
def test_extract_schema_requires_all_fields():
    required = set(EXTRACT_SCHEMA["input_schema"]["required"])
    assert required == {"name", "age", "city"}, (
        "EXTRACT_SCHEMA's required fields regressed — downstream code assumes all three "
        "are always present without a None-check."
    )
```
</details>

## Key Takeaways
- Free-text "respond in JSON" prompts fail unpredictably — the exact behavior you don't want in a production system.
- Forcing a tool call with `tool_choice={"type": "tool", "name": ...}` guarantees a schema-shaped response; Pydantic then validates the values on top of that shape.
- The validate → repair loop feeds validation errors back to the model, resolving most edge cases in one extra round-trip instead of failing the whole request.
- Structured outputs are the boundary between "the model said something" and "your code can trust it" — treat that boundary explicitly, everywhere an LLM output feeds a program.

## What's Next
Notebook 29 wraps a structured-output-producing LLM feature in an actual served API — streaming, timeouts, retries, and fallback models.
